In [1]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

base = os.path.expanduser("~/IDX-Exchange-Internship-Data_Science-Summer2026/california")

files = glob.glob(os.path.join(base, "CRMLSSold*.csv"))

print("Base folder:", base)
print("Folder exists:", os.path.exists(base))
print("Files found:", len(files))
print(files[:5])

Base folder: /Users/nguyenanh/IDX-Exchange-Internship-Data_Science-Summer2026/california
Folder exists: True
Files found: 30
['/Users/nguyenanh/IDX-Exchange-Internship-Data_Science-Summer2026/california/CRMLSSold202404_filled.csv', '/Users/nguyenanh/IDX-Exchange-Internship-Data_Science-Summer2026/california/CRMLSSold202409.csv', '/Users/nguyenanh/IDX-Exchange-Internship-Data_Science-Summer2026/california/CRMLSSold202408.csv', '/Users/nguyenanh/IDX-Exchange-Internship-Data_Science-Summer2026/california/CRMLSSold202501_filled.csv', '/Users/nguyenanh/IDX-Exchange-Internship-Data_Science-Summer2026/california/CRMLSSold20220101_20231231_filled.csv']


In [2]:
# Load all files
dfs = []
for f in sorted(files):
    temp = pd.read_csv(f, low_memory=False)
    temp["_source_file"] = os.path.basename(f)  # track which file each row came from
    print(f"{os.path.basename(f):50s} → {len(temp):>8,} rows, {temp.shape[1]} cols")
    dfs.append(temp)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows: {len(df_raw):,}")
print(f"Total columns: {df_raw.shape[1]}")

CRMLSSold20220101_20231231_filled.csv              →  157,828 rows, 81 cols
CRMLSSold202401_filled.csv                         →   17,958 rows, 81 cols
CRMLSSold202402_filled.csv                         →   19,925 rows, 81 cols
CRMLSSold202403_filled.csv                         →   23,276 rows, 81 cols
CRMLSSold202404_filled.csv                         →   24,640 rows, 81 cols
CRMLSSold202405_filled.csv                         →   26,487 rows, 81 cols
CRMLSSold202406_filled.csv                         →   24,328 rows, 81 cols
CRMLSSold202407_filled.csv                         →   26,240 rows, 81 cols
CRMLSSold202408.csv                                →   24,558 rows, 79 cols
CRMLSSold202409.csv                                →   21,267 rows, 79 cols
CRMLSSold202410.csv                                →   23,274 rows, 79 cols
CRMLSSold202411.csv                                →   20,279 rows, 79 cols
CRMLSSold202412.csv                                →   20,241 rows, 79 cols
CRMLSSold202

### **Filter to Residential + SingleFamilyResidence**

In [3]:
df = df_raw[
    (df_raw["PropertyType"] == "Residential") &
    (df_raw["PropertySubType"] == "SingleFamilyResidence")
].copy()

print("Rows before property filter:", f"{len(df_raw):,}")
print("Rows after property filter:", f"{len(df):,}")
print("Rows removed:", f"{len(df_raw) - len(df):,}")

Rows before property filter: 794,271
Rows after property filter: 399,157
Rows removed: 395,114


### **Convert Date Column**

In [4]:
# Automatically find columns that look like date columns
date_cols = [col for col in df.columns if "date" in col.lower()]

print("Date columns found:")
print(date_cols)

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")
    print(col, "missing after conversion:", df[col].isna().sum())

df["close_year_month"] = df["CloseDate"].dt.to_period("M").astype(str)

df[["CloseDate", "close_year_month"]].head()

Date columns found:
['CloseDate', 'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate']
CloseDate missing after conversion: 0
ContractStatusChangeDate missing after conversion: 8243
PurchaseContractDate missing after conversion: 143
ListingContractDate missing after conversion: 0


,CloseDate,close_year_month
3,2022-01-04,2022-01
6,2022-01-10,2022-01
7,2022-03-23,2022-03
12,2022-01-10,2022-01
14,2022-01-19,2022-01


I automatically identified date-related columns by selecting columns with "date" in the column name. After converting them to datetime format, CloseDate and ListingContractDate had no missing values, while PurchaseContractDate and ContractStatusChangeDate had some missing values. I created a close_year_month column from CloseDate so the dataset can be split by month for the time-based train/test split.

In [5]:
before = len(df)

df = df[df["ClosePrice"].notna()]
df = df[df["ClosePrice"] > 0]
df = df[df["CloseDate"].notna()]

after = len(df)

print("Rows before target/date filtering:", f"{before:,}")
print("Rows after target/date filtering:", f"{after:,}")
print("Rows removed:", f"{before - after:,}")

Rows before target/date filtering: 399,157
Rows after target/date filtering: 399,154
Rows removed: 3


After filtering out rows with missing or non-positive ClosePrice and missing CloseDate, only 3 rows were removed. This shows that the target variable and close date are mostly complete for the residential single-family dataset.

### **Remove Duplicate Records**

In [6]:
duplicate_count = df["ListingKey"].duplicated().sum()
print("Duplicate ListingKey rows before removal:", duplicate_count)

before = len(df)

df = df.sort_values("CloseDate")
df = df.drop_duplicates(subset="ListingKey", keep="last")

after = len(df)

print("Rows before duplicate removal:", f"{before:,}")
print("Rows after duplicate removal:", f"{after:,}")
print("Rows removed:", f"{before - after:,}")
print("Duplicate ListingKey rows after removal:", df["ListingKey"].duplicated().sum())

Duplicate ListingKey rows before removal: 275
Rows before duplicate removal: 399,154
Rows after duplicate removal: 398,879
Rows removed: 275
Duplicate ListingKey rows after removal: 0


### **Feature Selection**

In [7]:
# Get all numeric columns
all_numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()

# Check missing values for all numeric columns
numeric_missing_summary = pd.DataFrame({
    "missing_count": df[all_numeric_cols].isna().sum(),
    "missing_percent": df[all_numeric_cols].isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

numeric_missing_summary

,missing_count,missing_percent
MiddleOrJuniorSchoolDistrict,398879,100.000000
FireplacesTotal,398879,100.000000
ElementarySchoolDistrict,398879,100.000000
TaxAnnualAmount,398879,100.000000
CoveredSpaces,398879,100.000000
TaxYear,398879,100.000000
AboveGradeFinishedArea,398879,100.000000
BelowGradeFinishedArea,396554,99.417116
BuildingAreaTotal,374219,93.817674
BuyerAgencyCompensation,278018,69.699834


In [8]:
# Columns to exclude because they are target, ID-like, or not useful as predictors
exclude_numeric_cols = [
    "ClosePrice",
    "ListingKey",
    "ListingKeyNumeric",
    "StreetNumberNumeric"
]

# Keep numeric columns with less than 50% missing
usable_numeric_cols = numeric_missing_summary[
    numeric_missing_summary["missing_percent"] < 50
].index.tolist()

numeric_features = [
    col for col in usable_numeric_cols
    if col not in exclude_numeric_cols
]

print("Selected numeric features:")
print(numeric_features)
print("Number of numeric features:", len(numeric_features))

Selected numeric features:
['MainLevelBedrooms', 'AssociationFee', 'Stories', 'GarageSpaces', 'ParkingTotal', 'LotSizeAcres', 'LotSizeSquareFeet', 'LotSizeArea', 'OriginalListPrice', 'YearBuilt', 'LivingArea', 'Longitude', 'Latitude', 'BathroomsTotalInteger', 'BedroomsTotal', 'DaysOnMarket', 'ListPrice']
Number of numeric features: 17


Before selecting numeric features, I reviewed the missing percentage for every numeric column. I excluded columns that were mostly missing, such as FireplacesTotal, TaxAnnualAmount, CoveredSpaces, and BuildingAreaTotal. I also excluded target or ID-like columns such as ClosePrice, ListingKey, ListingKeyNumeric, and StreetNumberNumeric. For the remaining numeric columns with acceptable missingness, I kept them as candidate numeric features and handled missing values later using missing-value flags and median imputation.

In [9]:
# Get all categorical/object columns
all_categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("All categorical columns:")
print(all_categorical_cols)
print("Number of categorical columns:", len(all_categorical_cols))

All categorical columns:
['Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'ListAgentEmail', 'ListAgentFirstName', 'ListAgentLastName', 'UnparsedAddress', 'PropertyType', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'AssociationFeeFrequency', 'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'BuilderName', 'PropertySubType', 'SubdivisionName', 'BuyerOfficeAOR', 'BuyerAgencyCompensationType', 'ListingId', 'City', 'CoBuyerAgentFirstName', 'BusinessType', 'StateOrProvince', 'MiddleOrJuniorSchool', 'FireplaceYN', 'HighSchool', 'Levels', 'LotSizeDimensions', 'NewConstructionYN', 'HighSchoolDistrict', 'PostalCode', 'latfilled', 'lonfilled', '_source_file', 'BuyerAgentAOR', 'ListAgentAOR', 'close_year_month']
Number of categorical columns: 49


In [10]:
categorical_summary = pd.DataFrame({
    "missing_count": df[all_categorical_cols].isna().sum(),
    "missing_percent": df[all_categorical_cols].isna().mean() * 100,
    "unique_count": df[all_categorical_cols].nunique(dropna=True)
}).sort_values("missing_percent", ascending=False)

categorical_summary

,missing_count,missing_percent,unique_count
BusinessType,398879,100.000000,0
WaterfrontYN,398704,99.956127,1
BasementYN,389140,97.558407,1
BuilderName,380131,95.299828,3320
LotSizeDimensions,374377,93.857285,11324
CoBuyerAgentFirstName,363247,91.066965,4385
ElementarySchool,343471,86.109071,1386
MiddleOrJuniorSchool,342887,85.962660,663
HighSchool,326938,81.964205,516
CoListAgentFirstName,308801,77.417212,6567


In [11]:
# Keep categorical columns with less than 50% missing
# and not too many unique values
usable_categorical_cols = categorical_summary[
    (categorical_summary["missing_percent"] < 50) &
    (categorical_summary["unique_count"] <= 1000)
].index.tolist()

usable_categorical_cols

['Flooring',
 'BuyerAgentAOR',
 'ListAgentAOR',
 'HighSchoolDistrict',
 'AttachedGarageYN',
 'Levels',
 'PoolPrivateYN',
 'NewConstructionYN',
 'ViewYN',
 'BuyerOfficeAOR',
 'FireplaceYN',
 'StateOrProvince',
 '_source_file',
 'MlsStatus',
 'PropertyType',
 'PropertySubType',
 'CountyOrParish',
 'close_year_month']

In [12]:
categorical_features = [
    "CountyOrParish",
    "City",
    "PostalCode",
    "HighSchoolDistrict",
    "Flooring",
    "AttachedGarageYN",
    "Levels",
    "PoolPrivateYN",
    "NewConstructionYN",
    "ViewYN",
    "FireplaceYN"
]

categorical_features = [col for col in categorical_features if col in df.columns]

print("Selected categorical features:")
print(categorical_features)
print("Number of categorical features:", len(categorical_features))

Selected categorical features:
['CountyOrParish', 'City', 'PostalCode', 'HighSchoolDistrict', 'Flooring', 'AttachedGarageYN', 'Levels', 'PoolPrivateYN', 'NewConstructionYN', 'ViewYN', 'FireplaceYN']
Number of categorical features: 11


Before selecting categorical features, I reviewed each object-type column based on missing percentage and number of unique values. I dropped columns that were mostly missing, such as BusinessType, WaterfrontYN, BasementYN, and BuilderName. I also avoided columns with extremely high cardinality, such as UnparsedAddress, ListingId, agent names, agent emails, and office names, because one-hot encoding them would create too many sparse columns. I kept meaningful location and property-related categorical features such as CountyOrParish, City, PostalCode, HighSchoolDistrict, Flooring, garage, pool, new construction, view, and fireplace indicators.

In [13]:
target_col = "ClosePrice"

id_cols = [
    "ListingKey",
    "CloseDate",
    "close_year_month",
    "_source_file"
]

# Numeric columns selected after checking missingness
numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
    "YearBuilt",
    "DaysOnMarket",
    "GarageSpaces",
    "ParkingTotal",
    "OriginalListPrice",
    "ListPrice",
    "Latitude",
    "Longitude",
    "Stories",
    "AssociationFee",
    "MainLevelBedrooms"
]

# Categorical columns selected after checking missingness + unique counts
categorical_features = [
    "CountyOrParish",
    "City",
    "PostalCode",
    "HighSchoolDistrict",
    "Flooring",
    "AttachedGarageYN",
    "Levels",
    "PoolPrivateYN",
    "NewConstructionYN",
    "ViewYN",
    "FireplaceYN"
]

# Keep only columns that actually exist
id_cols = [col for col in id_cols if col in df.columns]
numeric_features = [col for col in numeric_features if col in df.columns]
categorical_features = [col for col in categorical_features if col in df.columns]

selected_cols = id_cols + [target_col] + numeric_features + categorical_features

df_model = df[selected_cols].copy()

print("Selected columns:")
print(selected_cols)

print("\nModeling dataset shape:", df_model.shape)

df_model.head()

Selected columns:
['ListingKey', 'CloseDate', 'close_year_month', '_source_file', 'ClosePrice', 'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'LotSizeAcres', 'LotSizeArea', 'YearBuilt', 'DaysOnMarket', 'GarageSpaces', 'ParkingTotal', 'OriginalListPrice', 'ListPrice', 'Latitude', 'Longitude', 'Stories', 'AssociationFee', 'MainLevelBedrooms', 'CountyOrParish', 'City', 'PostalCode', 'HighSchoolDistrict', 'Flooring', 'AttachedGarageYN', 'Levels', 'PoolPrivateYN', 'NewConstructionYN', 'ViewYN', 'FireplaceYN']

Modeling dataset shape: (398879, 33)


,ListingKey,CloseDate,close_year_month,_source_file,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,LotSizeAcres,...,City,PostalCode,HighSchoolDistrict,Flooring,AttachedGarageYN,Levels,PoolPrivateYN,NewConstructionYN,ViewYN,FireplaceYN
3494,543187090,2022-01-01,2022-01,CRMLSSold20220101_20231231_filled.csv,535000.0,2061.0,4.0,2.0,12232.0,NaN,...,Yucca Valley,92284,Morongo Unified,"Laminate,Tile",True,One,True,NaN,True,True
4253,542140363,2022-01-01,2022-01,CRMLSSold20220101_20231231_filled.csv,560000.0,1546.0,3.0,2.0,27225.0,NaN,...,Yucca Valley,92284,NaN,"Carpet,Tile",False,NaN,False,NaN,True,True
12194,497949539,2022-01-02,2022-01,CRMLSSold20220101_20231231_filled.csv,3300000.0,3085.0,3.0,3.0,42237.0,0.9696,...,Los Angeles,90068,Los Angeles Unified,NaN,False,One,False,False,True,True
4277,542133256,2022-01-03,2022-01,CRMLSSold20220101_20231231_filled.csv,691200.0,1242.0,4.0,2.0,6416.0,0.1473,...,La Puente,91744,Hacienda La Puente Unified,Wood,False,One,False,False,True,True
3237,543249572,2022-01-03,2022-01,CRMLSSold20220101_20231231_filled.csv,419000.0,1278.0,3.0,2.0,7665.0,0.1760,...,Yucaipa,92399,Yucaipa/Calimesa Unified,NaN,True,One,False,False,True,False


In [14]:
missing_summary = pd.DataFrame({
    "missing_count": df_model.isna().sum(),
    "missing_percent": df_model.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

missing_summary

,missing_count,missing_percent
MainLevelBedrooms,162403,40.714853
Flooring,142639,35.759967
AssociationFee,122230,30.643378
HighSchoolDistrict,103287,25.894319
Stories,54582,13.683849
AttachedGarageYN,46070,11.549869
Levels,41681,10.449535
PoolPrivateYN,40540,10.163483
NewConstructionYN,39536,9.911778
ViewYN,39513,9.906012


In [15]:
df_clean = df_model.copy()

# These numeric columns should be positive.
# If they are 0 or negative, treat them as missing.
positive_only_cols = [
    "LivingArea",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
    "YearBuilt",
    "OriginalListPrice",
    "ListPrice"
]

# These numeric columns can be 0, but should not be negative.
nonnegative_cols = [
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "DaysOnMarket",
    "GarageSpaces",
    "ParkingTotal",
    "Stories",
    "AssociationFee",
    "MainLevelBedrooms"
]

# Convert invalid numeric values to missing
for col in positive_only_cols:
    if col in df_clean.columns:
        df_clean.loc[df_clean[col] <= 0, col] = np.nan

for col in nonnegative_cols:
    if col in df_clean.columns:
        df_clean.loc[df_clean[col] < 0, col] = np.nan

# Add missing flags and impute numeric columns with median
for col in numeric_features:
    if col in df_clean.columns:
        df_clean[f"{col}_missing_flag"] = df_clean[col].isna().astype(int)
        median_value = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_value)
        print(f"{col}: median used for imputation = {median_value}")

# Fill categorical missing values with Unknown
for col in categorical_features:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype("object").fillna("Unknown")

# Check remaining missing values
df_clean.isna().sum().sort_values(ascending=False).head(20)

LivingArea: median used for imputation = 1800.0
BedroomsTotal: median used for imputation = 3.0
BathroomsTotalInteger: median used for imputation = 2.0
LotSizeSquareFeet: median used for imputation = 7250.0
LotSizeAcres: median used for imputation = 0.1666
LotSizeArea: median used for imputation = 7104.0
YearBuilt: median used for imputation = 1975.0
DaysOnMarket: median used for imputation = 16.0
GarageSpaces: median used for imputation = 2.0
ParkingTotal: median used for imputation = 2.0
OriginalListPrice: median used for imputation = 895000.0
ListPrice: median used for imputation = 883000.0
Latitude: median used for imputation = 34.099229
Longitude: median used for imputation = -118.059965
Stories: median used for imputation = 1.0
AssociationFee: median used for imputation = 0.0
MainLevelBedrooms: median used for imputation = 3.0


ListingKey                            0
LotSizeAcres_missing_flag             0
AttachedGarageYN                      0
Levels                                0
PoolPrivateYN                         0
NewConstructionYN                     0
ViewYN                                0
FireplaceYN                           0
LivingArea_missing_flag               0
BedroomsTotal_missing_flag            0
BathroomsTotalInteger_missing_flag    0
LotSizeSquareFeet_missing_flag        0
LotSizeArea_missing_flag              0
CloseDate                             0
YearBuilt_missing_flag                0
DaysOnMarket_missing_flag             0
GarageSpaces_missing_flag             0
ParkingTotal_missing_flag             0
OriginalListPrice_missing_flag        0
ListPrice_missing_flag                0
dtype: int64

In [16]:
if "LivingArea" in df_clean.columns:
    df_clean["price_per_sqft"] = df_clean["ClosePrice"] / df_clean["LivingArea"]

if "YearBuilt" in df_clean.columns:
    df_clean["sale_year"] = df_clean["CloseDate"].dt.year
    df_clean["home_age"] = df_clean["sale_year"] - df_clean["YearBuilt"]
    df_clean.loc[df_clean["home_age"] < 0, "home_age"] = np.nan
    df_clean["home_age_missing_flag"] = df_clean["home_age"].isna().astype(int)
    df_clean["home_age"] = df_clean["home_age"].fillna(df_clean["home_age"].median())

df_clean["log_ClosePrice"] = np.log1p(df_clean["ClosePrice"])

if "LivingArea" in df_clean.columns:
    df_clean["log_LivingArea"] = np.log1p(df_clean["LivingArea"])

if "LotSizeSquareFeet" in df_clean.columns:
    df_clean["log_LotSizeSquareFeet"] = np.log1p(df_clean["LotSizeSquareFeet"])

df_clean[[
    "ClosePrice",
    "LivingArea",
    "price_per_sqft",
    "YearBuilt",
    "sale_year",
    "home_age",
    "log_ClosePrice",
    "log_LivingArea",
    "log_LotSizeSquareFeet"
]].head()

,ClosePrice,LivingArea,price_per_sqft,YearBuilt,sale_year,home_age,log_ClosePrice,log_LivingArea,log_LotSizeSquareFeet
3494,535000.0,2061.0,259.582727,2007.0,2022,15.0,13.190024,7.631432,9.411892
4253,560000.0,1546.0,362.225097,1956.0,2022,66.0,13.235694,7.344073,10.211928
12194,3300000.0,3085.0,1069.692058,1953.0,2022,69.0,15.009433,8.034631,10.651076
4277,691200.0,1242.0,556.521739,1955.0,2022,67.0,13.446186,7.125283,8.766706
3237,419000.0,1278.0,327.856025,1955.0,2022,67.0,12.945629,7.153834,8.944550


| New feature | Created from | Why useful |
|---|---|---|
| `price_per_sqft` | `ClosePrice / LivingArea` | Compares home value relative to size. |
| `sale_year` | Year from `CloseDate` | Used to calculate the age of the home at the time of sale. |
| `home_age` | `sale_year - YearBuilt` | Measures how old the home was when it sold. |
| `home_age_missing_flag` | Missingness of `home_age` | Keeps track of whether `home_age` was originally missing or invalid. |
| `log_ClosePrice` | Log of `ClosePrice` | Reduces the effect of extreme home price outliers. |
| `log_LivingArea` | Log of `LivingArea` | Reduces the effect of very large home-size outliers. |
| `log_LotSizeSquareFeet` | Log of `LotSizeSquareFeet` | Reduces the effect of extremely large lot-size outliers. |

In [17]:
output_csv = "cleaned_residential_single_family_week3.csv"

df_clean.to_csv(output_csv, index=False)

print("Cleaned CSV saved as:", output_csv)
print("Rows:", f"{len(df_clean):,}")
print("Columns:", df_clean.shape[1])

Cleaned CSV saved as: cleaned_residential_single_family_week3.csv
Rows: 398,879
Columns: 57


#### **Requirement**
Use the newest month as the test set.

Use the months right before it as the training set.

In [18]:
available_months = sorted(df_clean["close_year_month"].dropna().unique()) # get all month and sorted from oldest to newest

print("Number of available months:", len(available_months))
print("First 10 months:", available_months[:10])
print("Last 10 months:", available_months[-10:])

test_month = available_months[-1] ## Choose the newest month 
print("\nMost recent month used as test set:", test_month)

Number of available months: 53
First 10 months: ['2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06', '2022-07', '2022-08', '2022-09', '2022-10']
Last 10 months: ['2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05']

Most recent month used as test set: 2026-05


In [19]:
def create_time_split(data, x_months, month_col="close_year_month"):
    months = sorted(data[month_col].dropna().unique())

    test_month = months[-1]

    months_before_test = months[:-1]

    train_months = months_before_test[-x_months:]

    train_df = data[data[month_col].isin(train_months)].copy()
    test_df = data[data[month_col] == test_month].copy()

    return train_df, test_df, train_months, test_month

In [20]:
X = 6

train_df, test_df, train_months, test_month = create_time_split(df_clean, X)

print("Example X:", X)
print("Training months:", train_months)
print("Test month:", test_month)
print("Train rows:", f"{len(train_df):,}")
print("Test rows:", f"{len(test_df):,}")

Example X: 6
Training months: ['2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04']
Test month: 2026-05
Train rows: 59,390
Test rows: 12,024


In [21]:
candidate_x_values = [3, 6, 9, 12, 18, 24]

split_summary = []

for x in candidate_x_values:
    try:
        temp_train, temp_test, temp_train_months, temp_test_month = create_time_split(df_clean, x)
        split_summary.append({
            "X_months": x,
            "train_start_month": temp_train_months[0],
            "train_end_month": temp_train_months[-1],
            "test_month": temp_test_month,
            "train_rows": len(temp_train),
            "test_rows": len(temp_test)
        })
    except ValueError:
        split_summary.append({
            "X_months": x,
            "train_start_month": None,
            "train_end_month": None,
            "test_month": None,
            "train_rows": None,
            "test_rows": None
        })

split_summary_df = pd.DataFrame(split_summary)
split_summary_df

,X_months,train_start_month,train_end_month,test_month,train_rows,test_rows
0,3,2026-02,2026-04,2026-05,31737,12024
1,6,2025-11,2026-04,2026-05,59390,12024
2,9,2025-08,2026-04,2026-05,94303,12024
3,12,2025-05,2026-04,2026-05,129867,12024
4,18,2024-11,2026-04,2026-05,190609,12024
5,24,2024-05,2026-04,2026-05,265955,12024


In [22]:
chosen_X = 12

train_df, test_df, train_months, test_month = create_time_split(df_clean, chosen_X)

print("Chosen X:", chosen_X)
print("Training months:", train_months)
print("Test month:", test_month)
print("Train rows:", f"{len(train_df):,}")
print("Test rows:", f"{len(test_df):,}")

Chosen X: 12
Training months: ['2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04']
Test month: 2026-05
Train rows: 129,867
Test rows: 12,024


I experimented with different training window lengths for X, including 3, 6, 9, 12, 18, and 24 months. The test set remains May 2026 because it is the most recent month in the dataset. As X increases, the training set becomes larger, but it also includes older market data. I selected X = 12 as a reasonable balance because it uses one full year of recent training data, giving 129,867 training rows while still keeping the training period close to the test month. The final choice can be validated later during model evaluation.

### **Encoding categorical features + scaling numerical features**

In [24]:
feature_numeric = numeric_features.copy()

In [25]:
for col in ["home_age", "log_LivingArea", "log_LotSizeSquareFeet"]:
    if col in df_clean.columns:
        feature_numeric.append(col)

In [29]:
# Finds every column whose name ends with missing_flag
missing_flag_cols = [col for col in df_clean.columns if col.endswith("_missing_flag")]
feature_numeric += missing_flag_cols # adds all missing-flag columns to the numeric feature list.


In [31]:
# Remove duplicates and keep only existing columns
feature_numeric = list(dict.fromkeys([col for col in feature_numeric if col in df_clean.columns]))

# Create categorical predictor list
feature_categorical = [col for col in categorical_features if col in df_clean.columns]


In [32]:
print("Numerical predictors:")
print(feature_numeric)

print("\nCategorical predictors:")
print(feature_categorical)

Numerical predictors:
['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'LotSizeAcres', 'LotSizeArea', 'YearBuilt', 'DaysOnMarket', 'GarageSpaces', 'ParkingTotal', 'OriginalListPrice', 'ListPrice', 'Latitude', 'Longitude', 'Stories', 'AssociationFee', 'MainLevelBedrooms', 'home_age', 'log_LivingArea', 'log_LotSizeSquareFeet', 'LivingArea_missing_flag', 'BedroomsTotal_missing_flag', 'BathroomsTotalInteger_missing_flag', 'LotSizeSquareFeet_missing_flag', 'LotSizeAcres_missing_flag', 'LotSizeArea_missing_flag', 'YearBuilt_missing_flag', 'DaysOnMarket_missing_flag', 'GarageSpaces_missing_flag', 'ParkingTotal_missing_flag', 'OriginalListPrice_missing_flag', 'ListPrice_missing_flag', 'Latitude_missing_flag', 'Longitude_missing_flag', 'Stories_missing_flag', 'AssociationFee_missing_flag', 'MainLevelBedrooms_missing_flag', 'home_age_missing_flag']

Categorical predictors:
['CountyOrParish', 'City', 'PostalCode', 'HighSchoolDistrict', 'Flooring', 'AttachedGarageYN', 

I created final numerical and categorical predictor lists before preprocessing. I started with the selected numeric features, added engineered numeric features such as home_age and log-transformed size variables, and included missing-value flags. I also created a categorical predictor list from the selected categorical features. I checked that all selected columns actually exist in the cleaned dataframe and removed duplicates to avoid errors during preprocessing.

In [33]:
# takes our training/test dataframe and keeps only the columns we want to use as predictors.
X_train_raw = train_df[feature_numeric + feature_categorical].copy()
X_test_raw = test_df[feature_numeric + feature_categorical].copy()

y_train = train_df["ClosePrice"].copy() # training target
y_test = test_df["ClosePrice"].copy() # test target

print("X_train_raw shape:", X_train_raw.shape)
print("X_test_raw shape:", X_test_raw.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train_raw shape: (129867, 49)
X_test_raw shape: (12024, 49)
y_train shape: (129867,)
y_test shape: (12024,)


In [34]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

In [36]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), feature_numeric),
        ("cat", encoder, feature_categorical)
    ],
    remainder="drop"
)

In [38]:
# Make sure all categorical columns are strings
for col in feature_categorical:
    X_train_raw[col] = X_train_raw[col].astype(str)
    X_test_raw[col] = X_test_raw[col].astype(str)

In [40]:
X_train_processed = preprocessor.fit_transform(X_train_raw)
X_test_processed = preprocessor.transform(X_test_raw)

print("Raw X_train shape:", X_train_raw.shape)
print("Processed X_train shape:", X_train_processed.shape)
print("Raw X_test shape:", X_test_raw.shape)
print("Processed X_test shape:", X_test_processed.shape)

Raw X_train shape: (129867, 49)
Processed X_train shape: (129867, 3901)
Raw X_test shape: (12024, 49)
Processed X_test shape: (12024, 3901)


In [41]:
train_csv = f"train_week3_X{chosen_X}_months.csv"
test_csv = f"test_week3_{test_month}.csv"

train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv, index=False)

print("Train CSV saved as:", train_csv)
print("Test CSV saved as:", test_csv)

Train CSV saved as: train_week3_X12_months.csv
Test CSV saved as: test_week3_2026-05.csv
